### Lang Chain @tools


---


In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("MONGO_CONNECTION_STRING"):
    print("Connection string for MONGO is not set. Please check your .env file.")
else:
    print("MONGO_CONNECTION_STRING loaded successfully.")

if not os.environ.get("OPENAI_API_KEY"):
    print("API KEY for OPENAI is not set. Please check your .env file.")
else:
    print("OPENAI_API_KEY loaded successfully.")

if not os.environ.get("GROQ_API_KEY"):
    print("API key for Groq is not set. Please check your .env file.")
else:
    print("API key loaded successfully.")

if not os.environ.get("OPENWEATHER_API_KEY"):
    print("API key for OpenWeather is not set. Please check your .env file.")
else:
    print("API key loaded successfully.")

print(os.getenv("MONGO_CONNECTION_STRING"))
print(os.getenv("OPENAI_API_KEY"))
print(os.environ.get("GROQ_API_KEY"))
print(os.environ.get("OPENWEATHER_API_KEY"))

In [ ]:
import pymongo

MONGO_CONNECTION_STRING = os.environ.get("MONGO_CONNECTION_STRING")
mongo_client = pymongo.MongoClient(MONGO_CONNECTION_STRING)

try:
    mongo_client.admin.command('ping')
    print("✅ Connected successfully!")
except Exception as e:
    print("❌ Connection failed:", e)

db = mongo_client["sample_airbnb"]
collection = db["listingsAndReviews"]


In [ ]:
import requests

class WeatherClient:
    def __init__(self, api_key: str):
        self.api_key = api_key
        self.base_url = "https://api.openweathermap.org/data/2.5"

    def get_current_weather(self, location: str) -> dict:
        """Get current weather for location"""
        endpoint = f"{self.base_url}/weather"
        params = {
            "q": location,
            "appid": self.api_key,
            "units": "metric"
        }
        response = requests.get(endpoint, params=params)
        return response.json()

    def get_forecast(self, location: str) -> dict:
        """Get weather forecast for location"""
        endpoint = f"{self.base_url}/forecast"
        params = {
            "q": location,
            "appid": self.api_key,
            "units": "metric"
        }
        response = requests.get(endpoint, params=params)
        return response.json()

In [ ]:
from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage, SystemMessage
from langchain.chat_models import init_chat_model
from typing import TypedDict, Literal
import json

class GraphState(TypedDict):
    location: str
    user_intent: str  # "weather" or "tourism"
    messages: list
    final_response: str

def parse_location_node(state: GraphState) -> GraphState:
    """Extract location and determine user intent"""
    llm = init_chat_model("llama-3.1-8b-instant", model_provider="groq")

    messages = [
        SystemMessage(content="""You are a helpful assistant.
        Analyze the user's message and extract:
        1. The location they're asking about
        2. Their intent: 'weather' if asking about weather/forecast, 'tourism' if asking about things to do/see

        Respond in JSON: {"location": "...", "intent": "weather|tourism"}"""),
        HumanMessage(content=state["messages"][-1])
    ]

    response = llm.invoke(messages)
    data = json.loads(response.content)

    return {
        "location": data["location"],
        "user_intent": data["intent"],
        "messages": state["messages"]
    }

def router_node(state: GraphState) -> Literal["weather_tool", "tourism_tool"]:
    """Route based on user intent"""
    if state["user_intent"] == "weather":
        return "weather_tool"
    else:
        return "tourism_tool"

def weather_tool_node(state: GraphState) -> GraphState:
    """Call weather API"""
    client = WeatherClient(api_key=os.environ.get("OPENWEATHER_API_KEY"))
    weather_data = client.get_current_weather(state["location"])

    response = f"""Weather in {state['location']}:
    Temperature: {weather_data['main']['temp']}°C
    Feels like: {weather_data['main']['feels_like']}°C
    Conditions: {weather_data['weather'][0]['description']}
    Humidity: {weather_data['main']['humidity']}%
    Wind speed: {weather_data['wind']['speed']} m/s
    """

    return {**state, "final_response": response}

def tourism_tool_node(state: GraphState) -> GraphState:
    """Query MongoDB for tourist spots in location"""

    pipeline = [
        {"$match": {
            "$or": [
                {"address.market": {"$regex": state["location"], "$options": "i"}}, #contains location, case-insensitive
                {"address.country": {"$regex": state["location"], "$options": "i"}}
            ]
        }},
        {"$sort": {"review_scores.review_scores_rating": -1}},
        {"$limit": 5},
        {"$project": {
            "name": 1,
            "property_type": 1,
            "address.street": 1,
            "review_scores.review_scores_rating": 1
        }}
    ]

    spots = list(db.listingsAndReviews.aggregate(pipeline))

    response = f"Top tourist spots near {state['location']}:\n\n"
    for i, spot in enumerate(spots, 1):
        response += f"{i}. {spot['name']} ({spot['property_type']})\n"
        response += f"   Rating: {spot.get('review_scores', {}).get('review_scores_rating', 'N/A')}\n"
        response += f"   {spot.get('address', {}).get('street', '')}\n\n"

    return {**state, "final_response": response}

# Build the graph
workflow = StateGraph(GraphState)
workflow.add_node("parse_location", parse_location_node)
workflow.add_node("weather_tool", weather_tool_node)
workflow.add_node("tourism_tool", tourism_tool_node)

# Add edges
workflow.set_entry_point("parse_location")
workflow.add_conditional_edges(
    "parse_location",
    router_node,
    {
        "weather_tool": "weather_tool",
        "tourism_tool": "tourism_tool"
    }
)
workflow.add_edge("weather_tool", END)
workflow.add_edge("tourism_tool", END)

app = workflow.compile()

In [ ]:
initial_state = {
    "location": "",
    "user_intent": "",
    "messages": ["What's the weather like in Paris?"],
    "final_response": ""
}

result = app.invoke(initial_state)
print(result["final_response"])

# Try tourism query
initial_state["messages"] = ["What are the must-see places in Portugal?"]
result = app.invoke(initial_state)
print(result["final_response"])